# 1、定义工具

In [3]:
from langchain_community.tools import TavilySearchResults
import os
import dotenv
dotenv.load_dotenv()
# 加载环境
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")
# 查询 Tavily 搜索 API
search = TavilySearchResults(max_results=1)

# 2、定义Retriever

In [6]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import dotenv

dotenv.load_dotenv()
# 1. 提供一个大模型
embedding_model = OpenAIEmbeddings()
# 2.加载HTML内容为一个文档对象
loader = WebBaseLoader("https://baike.baidu.com/item/%E6%81%90%E9%BE%99/139019")
docs = loader.load()
# 3.分割文档
splitter = RecursiveCharacterTextSplitter(
	chunk_size=1000,
	chunk_overlap=200
)
documents = splitter.split_documents(docs)
# 4.向量化 得到向量数据库对象
vector = FAISS.from_documents(documents, embedding_model)
# 5.创建检索器
retriever = vector.as_retriever()

# 3、创建工具、工具集

In [7]:
from langchain_core.tools import create_retriever_tool

# 创建一个工具来检索文档
retriever_tool = create_retriever_tool(
	retriever=retriever,
	name="Baidu_search",
	description="搜索百度百科",
)

# 构建一个工具集
tools = [search, retriever_tool]

# 4、语言模型调用工具

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 获取大模型
model = ChatOpenAI(model="gpt-4o-mini")
# 模型绑定工具
model_with_tools = model.bind_tools(tools)
# 根据输入自动调用工具
messages = [HumanMessage(content="今天西安天气怎么样")]
response = model_with_tools.invoke(messages)
print(f"ContentString: {response.content}")
print(f"ToolCalls: {response.tool_calls}")

ContentString: 
ToolCalls: [{'name': 'tavily_search_results_json', 'args': {'query': '西安今天天气'}, 'id': 'call_HLnVpO7cHGz1rijkHtBwpPvl', 'type': 'tool_call'}]


# 5、创建Agent程序（使用通用方式）

In [9]:
from langchain_classic import hub

prompt = hub.pull("hwchase17/openai-functions-agent")
print(prompt.messages)

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}), MessagesPlaceholder(variable_name='chat_history', optional=True), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}), MessagesPlaceholder(variable_name='agent_scratchpad')]


In [10]:
from langchain_classic.agents import create_tool_calling_agent, AgentExecutor

# 创建Agent对象
agent = create_tool_calling_agent(model, tools, prompt)
# 创建AgentExecutor对象
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# 6、运行Agent

In [11]:
print(agent_executor.invoke({"input": "恐龙的特征"}))



> Entering new AgentExecutor chain...

Invoking: `Baidu_search` with `{'query': '恐龙的特征'}`


[18]1993年7月，河南西峡恐龙蛋化石群的发现轰动世界。南阳盆地的恐龙蛋化石群种类多、数量多、分布广、保存完好，堪称世界之最。 [12]2012年4月，科学家在中国辽宁省西部发现一种新的暴龙类恐龙。这种被命名为华丽羽王龙的食肉恐龙是迄今发现的体型最大的带羽毛恐龙。 [13]2025年8月，研究团队系统梳理了广西侏罗—白垩纪地层中的四足动物化石记录，共识别出14处重要化石点。 [28]恐龙近亲在1861年发现的始祖鸟（早期认为它是最早的鸟类，近期研究认为其更可能属于原始恐爪龙类）与美颌龙形态相似，差别在于始祖鸟化石有明显的羽毛痕迹（美颌龙虽然也有羽毛，但它们很原始），事实上有相当一部分食肉恐龙具有原始羽毛，这显示恐龙与鸟类可能是近亲。自从1970年以来，许多研究报告指出现代鸟类极可能是兽脚亚目恐龙的直系后代，近年来的研究都倾向于把鸟类直接归入虚骨龙类（包括霸王龙、窃蛋龙、恐爪龙等）。鳄鱼则是另一群恐龙的现代近亲，但两者关系较非鸟恐龙与鸟类远。非鸟恐龙、鸟类、鳄鱼都属于初龙类演化支，该演化支首次出现于晚二叠纪，并在中三叠纪成为优势动物群。 [12]哺乳动物起源于爬行动物，它们的前身是“似哺乳类的爬行动物”，即兽孔目，早期则是“似爬行类的哺乳动物”，即哺乳型动物。 [12]中生代的爬行动物，大部分在中生代的末期灭绝了；一部分适应了变化的环境被保留下来，即现存的爬行动物（如龟鳖类、蛇类、鳄类等）；还有一部分沿着不同的进化方向，进化成了现今的鸟类和哺乳类。 [12]恐龙是介于冷血和温血之间的动物2014年6月，有关恐龙究竟是像鸟类和哺乳动物一样的温血动物，还是类似爬行动物、鱼类和两栖动物的冷血动物的问题终于有了答案——恐龙其实是介于冷血和温血之间的动物。 [12]“我们的结果显示恐龙所具有的生长速率和新陈代谢速率，既不是冷血生物体也不是温血生物体所具有的特征。它们既不像哺乳动物或者鸟类，也不像爬行动物或者鱼类，而是介于现代冷血动物和温血动物之间。简言之，它们的生理机能在现代社会并不常见。”美国亚利桑那大学进化生物学家和生态学家布莱恩·恩奎斯特说。墨西哥生物学家表示，正是这种中等程度的新陈

# 7、添加记忆

In [12]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
# 调取指定session_id对应的memory
def get_session_history(session_id: str) -> BaseChatMessageHistory:
	if session_id not in store:
		store[session_id] = ChatMessageHistory()
	return store[session_id]


agent_with_chat_history = RunnableWithMessageHistory(
	runnable=agent_executor,
	get_session_history=get_session_history,
	input_messages_key="input",
	history_messages_key="chat_history",
)
response = agent_with_chat_history.invoke(
	{"input": "Hi，我的名字是Cyber"},
	config={"configurable": {"session_id": "123"}},
)
print(response)



> Entering new AgentExecutor chain...
你好，Cyber！很高兴认识你。有任何我可以帮助你的吗？

> Finished chain.
{'input': 'Hi，我的名字是Cyber', 'chat_history': [], 'output': '你好，Cyber！很高兴认识你。有任何我可以帮助你的吗？'}


In [13]:
response = agent_with_chat_history.invoke(
    {"input": "我叫什么名字?"},
    config={"configurable": {"session_id": "123"}},
)
print(response)



> Entering new AgentExecutor chain...
你的名字是Cyber。有什么我可以帮助你的吗？

> Finished chain.
{'input': '我叫什么名字?', 'chat_history': [HumanMessage(content='Hi，我的名字是Cyber', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，Cyber！很高兴认识你。有任何我可以帮助你的吗？', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])], 'output': '你的名字是Cyber。有什么我可以帮助你的吗？'}


In [14]:
response = agent_with_chat_history.invoke(
    {"input": "我叫什么名字?"},
    config={"configurable": {"session_id": "4566"}},
)
print(response)



> Entering new AgentExecutor chain...
抱歉，我无法知道您的名字。如果您愿意，可以告诉我您的名字。

> Finished chain.
{'input': '我叫什么名字?', 'chat_history': [], 'output': '抱歉，我无法知道您的名字。如果您愿意，可以告诉我您的名字。'}
